# Scenario 04: Negative Testing

**QE Perspective:** APIs must reject invalid inputs and return clear errors. This notebook validates:

- **Invalid sampling parameters:** e.g. `temperature > 2.0` (or out-of-range) should be rejected or clamped with an appropriate error/behavior.
- **Non-existent MCP tool:** Requesting a tool that does not exist (e.g. `mcp::nonexistent_tool`) should result in a predictable error rather than silent failure.

Configuration: `BASE_URL` / `INFERENCE_MODEL`. Server at `http://localhost:8321`.


## Setup

Load base URL and model from environment; create the OGX client.


In [ ]:
import os
import scripts.helpers  # noqa: F401 — loads .env if present

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
base_url = base_url.rstrip("/")
if base_url.endswith("/v1"):
    base_url = base_url[:-3].rstrip("/")
model = os.environ.get("INFERENCE_MODEL")

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

from ogx_client import OgxClient

client = OgxClient(base_url=base_url)

## Negative: Invalid temperature (> 2.0)

Passing temperature above the valid range (e.g. 2.5) should result in an error (4xx). We assert that an exception is raised. (If your server clamps temperature instead of rejecting, relax the assertion to allow status completed.)


In [3]:
invalid_temp_raised = False
try:
    client.responses.create(
        model=model,
        input="Hello.",
        temperature=2.5,
    )
except Exception as e:
    invalid_temp_raised = True
    assert hasattr(e, "status_code") and e.status_code in (400, 422), (
        f"Expected 400/422, got {e}"
    )

assert invalid_temp_raised, (
    "Expected an error (or documented rejection) when temperature > 2.0"
)

## Negative: Non-existent MCP tool

Requesting a tool group that does not exist (e.g. `mcp::nonexistent_tool_xyz`) should result in an error. We assert that an exception is raised or the response status indicates failure.


In [4]:
nonexistent_tool_raised = False
try:
    r = client.responses.create(
        model=model,
        input="Call the calculator.",
        tools=["mcp::nonexistent_tool_xyz"],
        stream=False,
    )
    if getattr(r, "status", None) and str(r.status) != "completed":
        nonexistent_tool_raised = True
except Exception as e:
    nonexistent_tool_raised = True
    assert hasattr(e, "status_code") and e.status_code in (400, 404, 422), (
        f"Expected 4xx, got {e}"
    )

assert nonexistent_tool_raised, (
    "Expected an error or non-completed status when requesting a non-existent MCP tool"
)

## QE Assertions summary

- Invalid sampling (temp > 2.0): an exception is raised or the API returns an error.
- Non-existent MCP tool: an exception is raised or response status is not `completed`.
